# Lab 02 - PDF, DOCX, and API Data Processing
# Name: Zohaib Mukhtar
# Registration Number: SP24-BDS-054

In [1]:
import re
import csv
import json
import math
from collections import Counter

import pdfplumber
import docx
import pandas as pd
import requests

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords


ModuleNotFoundError: No module named 'pdfplumber'

## Task 1: Multi-Page PDF Extraction with Word Frequency Analysis

**(a)** Open a PDF file using PyPDF2 or pdfplumber and extract text from each page individually.
Store the result as a list of strings where each index represents one page. Print the total
number of pages extracted.

**(b)** Merge all page strings into a single string. Use `re.sub()` to remove all punctuation
and digits, convert to lowercase, and split on whitespace to produce a token list. Print the
total token count.

**(c)** Build a word frequency dictionary from the token list. Remove at least 20 common
stopwords. Print the top 10 most frequent words along with their counts in descending order.

**(d)** Accept a keyword from the user via `input()`. Search through the page-wise list, split
each page's text into sentences, and print every sentence containing the keyword along with
its page number. If no match is found, print an appropriate message.


In [2]:
# 1(a) - extract text page by page using pdfplumber

pdf_path = "sample_multipage.pdf"

pages_list = []   # each index = text of that page

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        page_text = page.extract_text()
        pages_list.append(page_text)

print("Total pages extracted:", len(pages_list))


Total pages extracted: 6


In [3]:
# 1(b) - merge everything into one big string and clean it up

full_text = " ".join(pages_list)

# remove punctuation
clean_text = re.sub(r'[^\w\s]', ' ', full_text)
# remove digits
clean_text = re.sub(r'\d+', ' ', clean_text)
# lowercase everything
clean_text = clean_text.lower()

tokens = clean_text.split()

print("Total token count:", len(tokens))


Total token count: 357


In [4]:
# 1(c) - word frequency, removing stopwords

# manual stopword list (30+ words)
my_stopwords = ['the','is','a','an','and','or','but','of','to','in','on','for','with',
                 'as','by','are','was','were','be','been','this','that','it','at','from',
                 'into','also','has','have','than','more','not','can','its']

word_freq = {}
for w in tokens:
    if w in my_stopwords:
        continue
    if w in word_freq:
        word_freq[w] += 1
    else:
        word_freq[w] = 1

# sort by count, highest first, take top 10
top_10 = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:10]

print("Top 10 most frequent words:")
for word, count in top_10:
    print(word, "->", count)


Top 10 most frequent words:
python -> 17
data -> 8
many -> 5
https -> 4
use -> 4
files -> 4
libraries -> 4
used -> 3
org -> 3
today -> 3


In [5]:
# 1(d) - search for a keyword the user types in, sentence by sentence, page by page

keyword = input("Enter a keyword to search for: ")

found_something = False

for page_num, page_text in enumerate(pages_list, start=1):
    # basic sentence split on . ! ?
    sentence_list = re.split(r'(?<=[.!?])\s+', page_text)
    for sentence in sentence_list:
        if keyword.lower() in sentence.lower():
            found_something = True
            print(f"Page {page_num}: {sentence.strip()}")

if not found_something:
    print("Sorry, no sentence with that keyword was found in the PDF.")


Sorry, no sentence with that keyword was found in the PDF.


## Task 2: Structured Page Metadata Extraction and CSV Export

**(a)** Open a multi-page PDF (minimum 5 pages). For each page, build a dictionary with
exactly these keys: `page_number`, `raw_text`, `word_count`, and `char_count` (excluding
spaces). Collect all dictionaries into a list called `pages_data` and print the first 2
entries.

**(b)** From `pages_data`, identify and print: (i) the page with the highest word count and
its full content, (ii) the page with the lowest non-zero word count, and (iii) the average
word count across all pages rounded to 2 decimal places. Handle pages that return zero words.

**(c)** Accept a search phrase via `input()`. Perform a case-insensitive search across all
pages. For every page containing the phrase, print its page number and how many times the
phrase appears. If no page contains the phrase, raise a custom exception named
`PhraseNotFoundError` and catch it gracefully.

**(d)** Using Python's built-in `csv` module (not pandas), write a file named
`pdf_summary.csv` with columns: `page_number`, `word_count`, `char_count`. Then re-read that
file and print its contents row by row to confirm the write was successful.


In [6]:
# 2(a) - build the pages_data list of dictionaries

pages_data = []

with pdfplumber.open(pdf_path) as pdf:
    for i, page in enumerate(pdf.pages, start=1):
        text = page.extract_text()
        if text is None:
            text = ""   # just in case a page has no text at all

        word_count = len(text.split())
        char_count = len(text.replace(" ", "").replace("\n", ""))

        page_dict = {
            "page_number": i,
            "raw_text": text,
            "word_count": word_count,
            "char_count": char_count
        }
        pages_data.append(page_dict)

print("First 2 entries of pages_data:")
for entry in pages_data[:2]:
    print(entry)
    print("---")


First 2 entries of pages_data:
{'page_number': 1, 'raw_text': 'Python Programming Basics\nPython is a popular programming language used for data science, web development, and automation.\nIt was created by Guido van Rossum and released in 1991. Visit https://www.python.org for more info.\nMany students learn python because it is easy to read and has simple syntax rules.\nThere are 2 major versions currently in use today across many companies worldwide.', 'word_count': 61, 'char_count': 329}
---
{'page_number': 2, 'raw_text': 'Data Structures Overview\nLists, tuples, dictionaries, and sets are the core built in data structures in python.\nA list can store 100 items or more without any real limit on its size at all.\nDictionaries store key value pairs and are extremely useful for fast lookups in code.\nFor more details check https://docs.python.org/3/tutorial/datastructures.html today.', 'word_count': 54, 'char_count': 307}
---


In [7]:
# 2(b) - highest / lowest word count pages + average

max_page = max(pages_data, key=lambda p: p["word_count"])

# only look at pages that actually have words for the "lowest non-zero" part
nonzero_pages = [p for p in pages_data if p["word_count"] > 0]
min_page = min(nonzero_pages, key=lambda p: p["word_count"])

total_words = sum(p["word_count"] for p in pages_data)
avg_word_count = round(total_words / len(pages_data), 2)

print("Page with HIGHEST word count -> page", max_page["page_number"], "with", max_page["word_count"], "words")
print("Full content of that page:")
print(max_page["raw_text"])
print()
print("Page with LOWEST non-zero word count -> page", min_page["page_number"], "with", min_page["word_count"], "words")
print()
print("Average word count across all pages:", avg_word_count)


Page with HIGHEST word count -> page 3 with 62 words
Full content of that page:
Working with Files
Reading and writing files is a common task in python programs across many domains.
The open() function is used with modes like read, write, and append for files.
It is good practice to close files after use or use the with statement instead of manual closing.
In 2020 many tutorials started recommending the with statement as the default approach.

Page with LOWEST non-zero word count -> page 2 with 54 words

Average word count across all pages: 58.5


In [8]:
# 2(c) - search phrase, with a custom exception if nothing is found

class PhraseNotFoundError(Exception):
    pass

search_phrase = input("Enter a phrase to search for: ")

matches_found = False

try:
    for p in pages_data:
        text_lower = p["raw_text"].lower()
        count = text_lower.count(search_phrase.lower())
        if count > 0:
            matches_found = True
            print(f"Page {p['page_number']}: phrase appears {count} time(s)")

    if not matches_found:
        raise PhraseNotFoundError(f"'{search_phrase}' was not found in any page of the PDF.")

except PhraseNotFoundError as e:
    print("Caught an error:", e)


Caught an error: 'it is me' was not found in any page of the PDF.


In [9]:
# 2(d) - write pdf_summary.csv using the csv module (no pandas here)

with open("pdf_summary.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["page_number", "word_count", "char_count"])   # header
    for p in pages_data:
        writer.writerow([p["page_number"], p["word_count"], p["char_count"]])

print("CSV file written. Reading it back:")

with open("pdf_summary.csv", "r") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)


CSV file written. Reading it back:
['page_number', 'word_count', 'char_count']
['1', '61', '329']
['2', '54', '307']
['3', '62', '305']
['4', '58', '344']
['5', '61', '324']
['6', '55', '338']


## Task 3: PDF Text Preprocessing Pipeline for NLP

**(a)** Open any PDF file and extract all text into one string. Apply the following cleaning
steps in order using `re.sub()`: (i) remove all URLs, (ii) remove all digits, (iii) remove all
non-alphabetic characters except spaces and full stops, (iv) collapse multiple consecutive
spaces into a single space. Print the character count before and after cleaning.

**(b)** Convert the cleaned text to lowercase. Split into a list of sentences using
`re.split(r'[.!?]', text)`. Strip each sentence of leading and trailing whitespace. Remove
empty strings. Print the sentence count before and after removing empty strings.

**(c)** Tokenize each sentence into words. Remove English stopwords using
`nltk.corpus.stopwords.words('english')`. Discard any sentence that has fewer than 5 tokens
remaining after stopword removal. Print the sentence count before and after this filter and
display 3 sample sentences.

**(d)** Write all remaining sentences to a file named `nlp_ready_output.txt`, one sentence per
line. Re-read the file and print: (i) total lines written, (ii) the longest sentence by
character length, and (iii) the shortest sentence by character length.


In [10]:
# 3(a) - get all text from the pdf into one string, then clean it step by step

with pdfplumber.open(pdf_path) as pdf:
    raw_text_all = ""
    for page in pdf.pages:
        t = page.extract_text()
        if t:
            raw_text_all += t + " "

chars_before = len(raw_text_all)

# (i) remove urls
cleaned = re.sub(r'https?://\S+', ' ', raw_text_all)
# (ii) remove digits
cleaned = re.sub(r'\d+', ' ', cleaned)
# (iii) remove everything except letters, spaces and full stops
cleaned = re.sub(r'[^a-zA-Z\s.]', ' ', cleaned)
# (iv) collapse multiple spaces into one
cleaned = re.sub(r' +', ' ', cleaned)

chars_after = len(cleaned)

print("Character count BEFORE cleaning:", chars_before)
print("Character count AFTER cleaning:", chars_after)


Character count BEFORE cleaning: 2298
Character count AFTER cleaning: 2139


In [11]:
# 3(b) - lowercase + split into sentences

cleaned_lower = cleaned.lower()

sentence_list_raw = re.split(r'[.!?]', cleaned_lower)

# strip whitespace off each one
sentence_list_stripped = [s.strip() for s in sentence_list_raw]

sentence_count_before = len(sentence_list_stripped)

# remove empty strings
sentence_list = [s for s in sentence_list_stripped if s != ""]

sentence_count_after = len(sentence_list)

print("Sentence count before removing empty ones:", sentence_count_before)
print("Sentence count after removing empty ones:", sentence_count_after)


Sentence count before removing empty ones: 26
Sentence count after removing empty ones: 25


In [12]:
# 3(c) - tokenize, remove stopwords, drop short sentences

stop_words = set(stopwords.words('english'))

sentence_count_before_filter = len(sentence_list)

filtered_sentences = []
for sentence in sentence_list:
    words = sentence.split()
    words_no_stop = [w for w in words if w not in stop_words]
    if len(words_no_stop) >= 5:
        filtered_sentences.append(sentence)

sentence_count_after_filter = len(filtered_sentences)

print("Sentence count BEFORE stopword/length filter:", sentence_count_before_filter)
print("Sentence count AFTER stopword/length filter:", sentence_count_after_filter)
print()
print("3 sample sentences:")
for s in filtered_sentences[:3]:
    print("-", s)


Sentence count BEFORE stopword/length filter: 25
Sentence count AFTER stopword/length filter: 23

3 sample sentences:
- python programming basics
python is a popular programming language used for data science web development and automation
- it was created by guido van rossum and released in
- many students learn python because it is easy to read and has simple syntax rules


In [13]:
# 3(d) - save to txt file, then read it back and check longest/shortest

with open("nlp_ready_output.txt", "w") as f:
    for s in filtered_sentences:
        f.write(s + "\n")

with open("nlp_ready_output.txt", "r") as f:
    lines = [line.strip() for line in f.readlines()]

print("Total lines written:", len(lines))

longest = max(lines, key=len)
shortest = min(lines, key=len)

print("Longest sentence:", longest)
print("Shortest sentence:", shortest)


Total lines written: 29
Longest sentence: it is good practice to close files after use or use the with statement instead of manual closing
Shortest sentence: working with files


## Task 4: DOCX Paragraph Classification and Statistical Analysis

**(a)** Open any `.docx` file using python-docx. For each paragraph, store its text and
`style.name` in a list of dictionaries. Print the total paragraph count and all unique style
names found in the document.

**(b)** Classify all paragraphs into exactly three groups: (i) Headings - style name starts
with "Heading", (ii) Body - style is "Normal" or contains "Body", (iii) Other/Empty -
everything else including blank paragraphs. Print the count of each group.

**(c)** For every Body paragraph with more than 5 words, compute its word count and character
count excluding spaces. Print: (i) the paragraph with the highest word count, (ii) the
paragraph with the lowest word count above 5 words, and (iii) the mean word count across all
qualifying body paragraphs rounded to 2 decimal places.

**(d)** Build a nested dictionary structured as `{ style_name: { "count": N, "texts": [list] } }`
for all styles found. Sort the outer dictionary by count in descending order. If a style has
more than 2 paragraphs, show only the first 2 texts. Pretty-print the result.


In [14]:
# 4(a) - read paragraphs and their style names

doc_a = docx.Document("document_A.docx")

paragraph_data = []
for para in doc_a.paragraphs:
    paragraph_data.append({
        "text": para.text,
        "style": para.style.name
    })

print("Total paragraph count:", len(paragraph_data))

unique_styles = set(p["style"] for p in paragraph_data)
print("Unique style names found:", unique_styles)


Total paragraph count: 14
Unique style names found: {'Heading 2', 'Normal', 'Heading 1'}


In [15]:
# 4(b) - classify into Headings / Body / Other-Empty

heading_count = 0
body_count = 0
other_count = 0

for p in paragraph_data:
    style = p["style"]
    if style.startswith("Heading"):
        heading_count += 1
    elif style == "Normal" or "Body" in style:
        body_count += 1
    else:
        other_count += 1

print("Headings:", heading_count)
print("Body:", body_count)
print("Other/Empty:", other_count)


Headings: 5
Body: 9
Other/Empty: 0


In [16]:
# 4(c) - stats on body paragraphs that have more than 5 words

qualifying_paragraphs = []

for p in paragraph_data:
    style = p["style"]
    if style == "Normal" or "Body" in style:
        word_count = len(p["text"].split())
        if word_count > 5:
            char_count = len(p["text"].replace(" ", ""))
            qualifying_paragraphs.append({
                "text": p["text"],
                "word_count": word_count,
                "char_count": char_count
            })

highest = max(qualifying_paragraphs, key=lambda x: x["word_count"])
lowest = min(qualifying_paragraphs, key=lambda x: x["word_count"])
mean_wc = round(sum(x["word_count"] for x in qualifying_paragraphs) / len(qualifying_paragraphs), 2)

print("Highest word count paragraph:")
print(highest["text"], "(", highest["word_count"], "words )")
print()
print("Lowest word count paragraph (above 5 words):")
print(lowest["text"], "(", lowest["word_count"], "words )")
print()
print("Mean word count of qualifying body paragraphs:", mean_wc)


Highest word count paragraph:
Machine learning is a field of artificial intelligence that allows computers to learn patterns from data without being explicitly programmed for every single rule. ( 24 words )

Lowest word count paragraph (above 5 words):
Supervised learning uses labeled data where the correct answer is already known for every training example given to the model. ( 20 words )

Mean word count of qualifying body paragraphs: 21.57


In [17]:
# 4(d) - nested dictionary by style, sorted by count, sorted descending

style_dict = {}

for p in paragraph_data:
    style = p["style"]
    if style not in style_dict:
        style_dict[style] = {"count": 0, "texts": []}
    style_dict[style]["count"] += 1
    style_dict[style]["texts"].append(p["text"])

# only keep first 2 texts if more than 2 paragraphs of that style
for style in style_dict:
    if style_dict[style]["count"] > 2:
        style_dict[style]["texts"] = style_dict[style]["texts"][:2]

# sort by count descending -> make a new dict in that order
sorted_styles = dict(sorted(style_dict.items(), key=lambda item: item[1]["count"], reverse=True))

import pprint
pprint.pprint(sorted_styles)


{'Heading 1': {'count': 3,
               'texts': ['Introduction to Machine Learning',
                         'Common Algorithms']},
 'Heading 2': {'count': 2,
               'texts': ['Types of Learning', 'Evaluation Metrics']},
 'Normal': {'count': 9,
            'texts': ['Machine learning is a field of artificial intelligence '
                      'that allows computers to learn patterns from data '
                      'without being explicitly programmed for every single '
                      'rule.',
                      'This document gives a short and simple overview of the '
                      'basic ideas behind machine learning models and how they '
                      'are trained.']}}


## Task 5: DOCX Table Extraction and Pandas Integration

**(a)** Open a `.docx` file that contains at least one table. Iterate through `doc.tables` and
extract each table as a list of lists: `[[row1col1, row1col2, ...], [row2col1, ...]]`. Print
the total number of tables found and the dimensions (rows x columns) of each.

**(b)** Take the first table's data, treat the first row as the header and the remaining rows
as data, and create a pandas DataFrame. Print the full DataFrame. Then count and print the
number of empty or None cells per column.

**(c)** Extract all paragraphs that appear before the first table in the document by iterating
through `doc.element.body` children and stopping once a `w:tbl` element is encountered. Store
these paragraphs in a list and print their count.

**(d)** Write both the pre-table paragraphs and the first table's data into a single file named
`docx_extracted.txt`. The file must have clearly labeled sections: a `--- PRE-TABLE PARAGRAPHS
---` header followed by the paragraphs, then a `--- TABLE DATA ---` header followed by rows
formatted as pipe-separated values, e.g., `col1 | col2 | col3`.


In [18]:
# 5(a) - extract every table in the docx as a list of lists

all_tables = []

for table in doc_a.tables:
    table_data = []
    for row in table.rows:
        row_data = [cell.text for cell in row.cells]
        table_data.append(row_data)
    all_tables.append(table_data)

print("Total tables found:", len(all_tables))

for i, t in enumerate(all_tables):
    rows = len(t)
    cols = len(t[0]) if rows > 0 else 0
    print(f"Table {i+1} dimensions: {rows} rows x {cols} columns")


Total tables found: 1
Table 1 dimensions: 4 rows x 3 columns


In [19]:
# 5(b) - first table -> pandas DataFrame

first_table = all_tables[0]

header = first_table[0]
data_rows = first_table[1:]

df_table = pd.DataFrame(data_rows, columns=header)
print(df_table)
print()

# count empty / None cells per column
# empty string counts as "empty" here too
empty_counts = {}
for col in df_table.columns:
    count = 0
    for val in df_table[col]:
        if val is None or val == "":
            count += 1
    empty_counts[col] = count

print("Empty/None cell counts per column:")
print(empty_counts)


           Algorithm           Type          Use Case
0  Linear Regression     Supervised  Price Prediction
1            K-Means   Unsupervised                  
2         Q-Learning  Reinforcement      Game Playing

Empty/None cell counts per column:
{'Algorithm': 0, 'Type': 0, 'Use Case': 1}


In [20]:
# 5(c) - paragraphs that appear BEFORE the first table

pre_table_paragraphs = []

body_children = doc_a.element.body

for child in body_children:
    tag = child.tag.split('}')[-1]   # strip the namespace part
    if tag == 'tbl':
        break   # stop as soon as we hit the first table
    if tag == 'p':
        # find the text inside this paragraph element manually
        text_parts = child.findall('.//{http://schemas.openxmlformats.org/wordprocessingml/2006/main}t')
        para_text = "".join([t.text if t.text else "" for t in text_parts])
        pre_table_paragraphs.append(para_text)

print("Number of paragraphs before the first table:", len(pre_table_paragraphs))
for p in pre_table_paragraphs:
    print("-", p)


Number of paragraphs before the first table: 14
- Introduction to Machine Learning
- Machine learning is a field of artificial intelligence that allows computers to learn patterns from data without being explicitly programmed for every single rule.
- This document gives a short and simple overview of the basic ideas behind machine learning models and how they are trained.
- Types of Learning
- Supervised learning uses labeled data where the correct answer is already known for every training example given to the model.
- Unsupervised learning
- Reinforcement learning trains an agent to take actions in an environment in order to maximize some notion of cumulative reward over time.
- Common Algorithms
- Some of the most common algorithms include linear regression, decision trees, random forests, and neural networks for various prediction tasks.
- 
- Evaluation Metrics
- Accuracy, precision, recall, and F1 score are commonly used metrics to evaluate how well a classification model is perfo

In [21]:
# 5(d) - write everything into docx_extracted.txt with labeled sections

with open("docx_extracted.txt", "w") as f:
    f.write("--- PRE-TABLE PARAGRAPHS ---\n")
    for p in pre_table_paragraphs:
        f.write(p + "\n")

    f.write("\n--- TABLE DATA ---\n")
    for row in first_table:
        f.write(" | ".join(row) + "\n")

print("File written. Here is what it contains:")
with open("docx_extracted.txt", "r") as f:
    print(f.read())


File written. Here is what it contains:
--- PRE-TABLE PARAGRAPHS ---
Introduction to Machine Learning
Machine learning is a field of artificial intelligence that allows computers to learn patterns from data without being explicitly programmed for every single rule.
This document gives a short and simple overview of the basic ideas behind machine learning models and how they are trained.
Types of Learning
Supervised learning uses labeled data where the correct answer is already known for every training example given to the model.
Unsupervised learning
Reinforcement learning trains an agent to take actions in an environment in order to maximize some notion of cumulative reward over time.
Common Algorithms
Some of the most common algorithms include linear regression, decision trees, random forests, and neural networks for various prediction tasks.

Evaluation Metrics
Accuracy, precision, recall, and F1 score are commonly used metrics to evaluate how well a classification model is performi

## Task 6: Multi-Document Comparison, Deduplication, and Frequency Report

**(a)** Open two different `.docx` files (Document A and Document B). From each, extract all
paragraphs whose style name starts with "Heading" into two separate lists. Using Python set
operations, print: (i) headings common to both documents, (ii) headings only in Document A,
and (iii) headings only in Document B.

**(b)** From each document, extract all "Normal" style paragraphs. Merge both lists. Remove
exact duplicate paragraphs by comparing stripped, lowercased text. Print the total count before
and after deduplication.

**(c)** Tokenize every paragraph in the deduplicated list into words (lowercase, punctuation
removed via `re.sub()`). Build a combined word frequency dictionary using
`collections.Counter`. Remove at least 15 stopwords. Display the top 20 most frequent words.

**(d)** Compute and print a summary containing: (i) total headings across both documents, (ii)
total unique body paragraphs after deduplication, (iii) total unique word types (vocabulary
size), (iv) average paragraph length in words rounded to 2 decimal places, and (v) the single
longest paragraph by word count from the combined set.


In [22]:
# 6(a) - headings from both docs, compared with sets

doc_b = docx.Document("document_B.docx")

headings_a = [p.text for p in doc_a.paragraphs if p.style.name.startswith("Heading")]
headings_b = [p.text for p in doc_b.paragraphs if p.style.name.startswith("Heading")]

set_a = set(headings_a)
set_b = set(headings_b)

common_headings = set_a & set_b
only_in_a = set_a - set_b
only_in_b = set_b - set_a

print("Headings common to both:", common_headings)
print("Headings only in Document A:", only_in_a)
print("Headings only in Document B:", only_in_b)


Headings common to both: {'Common Algorithms', 'Introduction to Machine Learning'}
Headings only in Document A: {'Evaluation Metrics', 'Conclusion', 'Types of Learning'}
Headings only in Document B: {'Deep Learning Basics', 'Applications'}


In [23]:
# 6(b) - Normal paragraphs from both, merged + deduplicated

normal_a = [p.text for p in doc_a.paragraphs if p.style.name == "Normal" and p.text.strip() != ""]
normal_b = [p.text for p in doc_b.paragraphs if p.style.name == "Normal" and p.text.strip() != ""]

merged_paragraphs = normal_a + normal_b
count_before = len(merged_paragraphs)

seen = set()
deduped_paragraphs = []
for para in merged_paragraphs:
    key = para.strip().lower()
    if key not in seen:
        seen.add(key)
        deduped_paragraphs.append(para)

count_after = len(deduped_paragraphs)

print("Total paragraph count BEFORE dedup:", count_before)
print("Total paragraph count AFTER dedup:", count_after)


Total paragraph count BEFORE dedup: 14
Total paragraph count AFTER dedup: 12


In [24]:
# 6(c) - tokenize everything, build a Counter, remove stopwords, show top 20

all_words = []
for para in deduped_paragraphs:
    clean_para = re.sub(r'[^\w\s]', '', para.lower())
    all_words.extend(clean_para.split())

word_counter = Counter(all_words)

# removing 18 stopwords here
basic_stopwords = ['the','is','a','an','and','or','but','of','to','in','on','for','with',
                    'as','are','that','this','from','it']

for sw in basic_stopwords:
    if sw in word_counter:
        del word_counter[sw]

top_20 = word_counter.most_common(20)
print("Top 20 most frequent words:")
for word, count in top_20:
    print(word, "->", count)


Top 20 most frequent words:
learning -> 9
data -> 5
machine -> 4
neural -> 4
networks -> 4
used -> 4
many -> 4
model -> 3
tasks -> 3
learn -> 2
patterns -> 2
every -> 2
single -> 2
how -> 2
uses -> 2
some -> 2
include -> 2
today -> 2
deep -> 2
image -> 2


In [25]:
# 6(d) - final summary

total_headings = len(headings_a) + len(headings_b)
total_unique_body_paragraphs = len(deduped_paragraphs)
vocab_size = len(word_counter)   # unique word types after stopword removal

word_counts_per_para = [len(p.split()) for p in deduped_paragraphs]
avg_para_length = round(sum(word_counts_per_para) / len(word_counts_per_para), 2)

longest_paragraph = max(deduped_paragraphs, key=lambda p: len(p.split()))

print("Total headings across both documents:", total_headings)
print("Total unique body paragraphs after dedup:", total_unique_body_paragraphs)
print("Vocabulary size (unique word types):", vocab_size)
print("Average paragraph length (words):", avg_para_length)
print("Longest paragraph:", longest_paragraph)


Total headings across both documents: 9
Total unique body paragraphs after dedup: 12
Vocabulary size (unique word types): 134
Average paragraph length (words): 19.75
Longest paragraph: Machine learning is a field of artificial intelligence that allows computers to learn patterns from data without being explicitly programmed for every single rule.


## Task 7: Multi-Endpoint API Data Joining and Analysis

**(a)** Send GET requests to both endpoints below. For each, verify the HTTP status code is
exactly 200; if not, raise a `ConnectionError` with the actual status code in the message.
Parse both responses as JSON lists.
`https://jsonplaceholder.typicode.com/users`
`https://jsonplaceholder.typicode.com/posts`

**(b)** Join the two datasets using `id` from users and `userId` from posts. Build a dictionary
structured as: `{ user_name: { "email": "...", "post_count": N, "titles": [list of post
titles] } }`. Print the result for the first 3 users only.

**(c)** From the joined dictionary, find the user who has written the most posts. Print their:
(i) full name, (ii) email address, (iii) company name, and (iv) all post titles numbered.

**(d)** Filter users where either of the following is true: (i) `company.bs` field contains the
word "synergies" (case-insensitive), or (ii) `address.city` has more than 8 characters. Print
the name, city, and post count for all matching users.

*(Note: this needs an internet connection since it hits a live API)*


In [26]:
# 7(a) - GET both endpoints and check the status code

users_url = "https://jsonplaceholder.typicode.com/users"
posts_url = "https://jsonplaceholder.typicode.com/posts"

users_response = requests.get(users_url)
if users_response.status_code != 200:
    raise ConnectionError(f"Users request failed with status code {users_response.status_code}")

posts_response = requests.get(posts_url)
if posts_response.status_code != 200:
    raise ConnectionError(f"Posts request failed with status code {posts_response.status_code}")

users_list = users_response.json()
posts_list = posts_response.json()

print("Users fetched:", len(users_list))
print("Posts fetched:", len(posts_list))


Users fetched: 10
Posts fetched: 100


In [27]:
# 7(b) - join users with their posts using id / userId

joined_data = {}

for user in users_list:
    user_id = user["id"]
    user_name = user["name"]

    # find all posts belonging to this user
    user_posts = [post for post in posts_list if post["userId"] == user_id]
    titles = [post["title"] for post in user_posts]

    joined_data[user_name] = {
        "email": user["email"],
        "post_count": len(user_posts),
        "titles": titles
    }

# print first 3 users only
first_3_names = list(joined_data.keys())[:3]
for name in first_3_names:
    print(name, "->", joined_data[name])
    print()


Leanne Graham -> {'email': 'Sincere@april.biz', 'post_count': 10, 'titles': ['sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'qui est esse', 'ea molestias quasi exercitationem repellat qui ipsa sit aut', 'eum et est occaecati', 'nesciunt quas odio', 'dolorem eum magni eos aperiam quia', 'magnam facilis autem', 'dolorem dolore est ipsam', 'nesciunt iure omnis dolorem tempora et accusantium', 'optio molestias id quia eum']}

Ervin Howell -> {'email': 'Shanna@melissa.tv', 'post_count': 10, 'titles': ['et ea vero quia laudantium autem', 'in quibusdam tempore odit est dolorem', 'dolorum ut in voluptas mollitia et saepe quo animi', 'voluptatem eligendi optio', 'eveniet quod temporibus', 'sint suscipit perspiciatis velit dolorum rerum ipsa laboriosam odio', 'fugit voluptas sed molestias voluptatem provident', 'voluptate et itaque vero tempora molestiae', 'adipisci placeat illum aut reiciendis qui', 'doloribus ad provident suscipit at']}

Clementine Bauch -> {'ema

In [28]:
# 7(c) - user with the most posts

top_user_name = max(joined_data, key=lambda name: joined_data[name]["post_count"])
top_user_info = joined_data[top_user_name]

# need the company name too, so grab it back from users_list
top_user_company = ""
for u in users_list:
    if u["name"] == top_user_name:
        top_user_company = u["company"]["name"]
        break

print("User with most posts:", top_user_name)
print("Email:", top_user_info["email"])
print("Company:", top_user_company)
print("Post titles:")
for i, title in enumerate(top_user_info["titles"], start=1):
    print(f"{i}. {title}")


User with most posts: Leanne Graham
Email: Sincere@april.biz
Company: Romaguera-Crona
Post titles:
1. sunt aut facere repellat provident occaecati excepturi optio reprehenderit
2. qui est esse
3. ea molestias quasi exercitationem repellat qui ipsa sit aut
4. eum et est occaecati
5. nesciunt quas odio
6. dolorem eum magni eos aperiam quia
7. magnam facilis autem
8. dolorem dolore est ipsam
9. nesciunt iure omnis dolorem tempora et accusantium
10. optio molestias id quia eum


In [29]:
# 7(d) - filter users by company.bs containing 'synergies' OR city longer than 8 chars

print("Matching users:")
for user in users_list:
    bs_text = user["company"]["bs"].lower()
    city = user["address"]["city"]

    condition1 = "synergies" in bs_text
    condition2 = len(city) > 8

    if condition1 or condition2:
        name = user["name"]
        post_count = joined_data[name]["post_count"]
        print(f"Name: {name}, City: {city}, Post count: {post_count}")


Matching users:
Name: Leanne Graham, City: Gwenborough, Post count: 10
Name: Ervin Howell, City: Wisokyburgh, Post count: 10
Name: Clementine Bauch, City: McKenziehaven, Post count: 10
Name: Patricia Lebsack, City: South Elvis, Post count: 10
Name: Chelsey Dietrich, City: Roscoeview, Post count: 10
Name: Mrs. Dennis Schulist, City: South Christy, Post count: 10
Name: Kurtis Weissnat, City: Howemouth, Post count: 10
Name: Nicholas Runolfsdottir V, City: Aliyaview, Post count: 10
Name: Glenna Reichert, City: Bartholomebury, Post count: 10
Name: Clementina DuBuque, City: Lebsackbury, Post count: 10


## Task 8: To-Do Completion Rate Analysis Across Users

**(a)** Fetch data from the endpoint below and parse it into a Python list. Validate that: (i)
the response status is 200, (ii) the parsed result is a list, and (iii) the list contains at
least one record. If any validation fails, print a meaningful error and exit. Print the total
number of todo records retrieved.
`https://jsonplaceholder.typicode.com/todos`

**(b)** Group todos by `userId`. For each user, compute and store: `{ userId: { "total": N,
"completed": N, "incomplete": N } }`. Print the result for all 10 users. For each user, verify
that `completed + incomplete == total` and print True or False accordingly.

**(c)** Compute the completion rate for each user as `completed / total * 100` rounded to 1
decimal place. Print: (i) the userId with the highest completion rate and its value, (ii) the
userId with the lowest completion rate and its value, and (iii) the overall average completion
rate across all users.

**(d)** Fetch `https://jsonplaceholder.typicode.com/users` and build a lookup dictionary
`{ userId: user_name }`. Print a formatted table with columns: `Rank | Name | Total Tasks |
Completed | Completion Rate %`, sorted by completion rate in descending order. Use f-strings
or `str.ljust()` to align columns neatly.

*(Note: this needs an internet connection since it hits a live API)*


In [30]:
# 8(a) - fetch todos and validate the response

todos_url = "https://jsonplaceholder.typicode.com/todos"
todos_response = requests.get(todos_url)

valid = True

if todos_response.status_code != 200:
    print("Error: status code was not 200, it was", todos_response.status_code)
    valid = False

todos_list = todos_response.json()

if not isinstance(todos_list, list):
    print("Error: response was not a list")
    valid = False

if len(todos_list) < 1:
    print("Error: no records found")
    valid = False

if not valid:
    raise SystemExit("Validation failed, stopping.")

print("Total todo records retrieved:", len(todos_list))


Total todo records retrieved: 200


In [31]:
# 8(b) - group todos by userId

user_todo_stats = {}

for todo in todos_list:
    uid = todo["userId"]
    if uid not in user_todo_stats:
        user_todo_stats[uid] = {"total": 0, "completed": 0, "incomplete": 0}

    user_todo_stats[uid]["total"] += 1
    if todo["completed"]:
        user_todo_stats[uid]["completed"] += 1
    else:
        user_todo_stats[uid]["incomplete"] += 1

for uid in sorted(user_todo_stats.keys()):
    stats = user_todo_stats[uid]
    print(uid, "->", stats)
    check = (stats["completed"] + stats["incomplete"]) == stats["total"]
    print("   completed + incomplete == total ?", check)


1 -> {'total': 20, 'completed': 11, 'incomplete': 9}
   completed + incomplete == total ? True
2 -> {'total': 20, 'completed': 8, 'incomplete': 12}
   completed + incomplete == total ? True
3 -> {'total': 20, 'completed': 7, 'incomplete': 13}
   completed + incomplete == total ? True
4 -> {'total': 20, 'completed': 6, 'incomplete': 14}
   completed + incomplete == total ? True
5 -> {'total': 20, 'completed': 12, 'incomplete': 8}
   completed + incomplete == total ? True
6 -> {'total': 20, 'completed': 6, 'incomplete': 14}
   completed + incomplete == total ? True
7 -> {'total': 20, 'completed': 9, 'incomplete': 11}
   completed + incomplete == total ? True
8 -> {'total': 20, 'completed': 11, 'incomplete': 9}
   completed + incomplete == total ? True
9 -> {'total': 20, 'completed': 8, 'incomplete': 12}
   completed + incomplete == total ? True
10 -> {'total': 20, 'completed': 12, 'incomplete': 8}
   completed + incomplete == total ? True


In [32]:
# 8(c) - completion rate per user

completion_rates = {}

for uid, stats in user_todo_stats.items():
    rate = round((stats["completed"] / stats["total"]) * 100, 1)
    completion_rates[uid] = rate

highest_uid = max(completion_rates, key=completion_rates.get)
lowest_uid = min(completion_rates, key=completion_rates.get)
overall_avg = round(sum(completion_rates.values()) / len(completion_rates), 1)

print("Highest completion rate: userId", highest_uid, "with", completion_rates[highest_uid], "%")
print("Lowest completion rate: userId", lowest_uid, "with", completion_rates[lowest_uid], "%")
print("Overall average completion rate:", overall_avg, "%")


Highest completion rate: userId 5 with 60.0 %
Lowest completion rate: userId 4 with 30.0 %
Overall average completion rate: 45.0 %


In [33]:
# 8(d) - lookup names, then print a nice formatted table sorted by completion rate

users_response_2 = requests.get("https://jsonplaceholder.typicode.com/users")
users_list_2 = users_response_2.json()

name_lookup = {}
for u in users_list_2:
    name_lookup[u["id"]] = u["name"]

# sort userIds by completion rate, highest first
sorted_uids = sorted(completion_rates, key=completion_rates.get, reverse=True)

print(f"{'Rank'.ljust(6)}{'Name'.ljust(22)}{'Total Tasks'.ljust(14)}{'Completed'.ljust(12)}{'Completion Rate %'}")
for rank, uid in enumerate(sorted_uids, start=1):
    name = name_lookup[uid]
    total = user_todo_stats[uid]["total"]
    completed = user_todo_stats[uid]["completed"]
    rate = completion_rates[uid]
    print(f"{str(rank).ljust(6)}{name.ljust(22)}{str(total).ljust(14)}{str(completed).ljust(12)}{rate}")


Rank  Name                  Total Tasks   Completed   Completion Rate %
1     Chelsey Dietrich      20            12          60.0
2     Clementina DuBuque    20            12          60.0
3     Leanne Graham         20            11          55.0
4     Nicholas Runolfsdottir V20            11          55.0
5     Kurtis Weissnat       20            9           45.0
6     Ervin Howell          20            8           40.0
7     Glenna Reichert       20            8           40.0
8     Clementine Bauch      20            7           35.0
9     Patricia Lebsack      20            6           30.0
10    Mrs. Dennis Schulist  20            6           30.0


## Task 9: Nested JSON Flattening, DataFrame Operations, and Comment Analysis

**(a)** Fetch data from `https://jsonplaceholder.typicode.com/users`. Each user contains nested
objects (`address` with `geo`, and `company`). Write a function `flatten_user(user: dict) ->
dict` that returns a single-level dictionary with exactly these keys: `id, name, username,
email, city, zipcode, lat, lng, company_name, company_catchphrase`. Apply this function to all
10 users and print the resulting flattened list.

**(b)** Create a pandas DataFrame from the flattened user list and display it in full. Then:
(i) cast the `lat` and `lng` columns to float, (ii) add a new column `distance_from_origin`
computed as `sqrt(lat^2 + lng^2)` using the `math` module applied row-by-row, and (iii) print
the user with the greatest distance from origin.

**(c)** Using the `lat` column, classify users into two groups: (i) Northern Hemisphere where
`lat > 0`, and (ii) Southern Hemisphere where `lat < 0`. Print the names of users in each
group. Then determine which hemisphere has the higher average `distance_from_origin`.

**(d)** Fetch both endpoints below. For each post, count its number of comments by matching
`postId` in comments against `id` in posts. Build a sorted list of tuples: `(post_id,
post_title[:40], comment_count)` in descending order by comment count. Print the top 5
most-commented and bottom 5 least-commented posts.
`https://jsonplaceholder.typicode.com/posts`
`https://jsonplaceholder.typicode.com/comments`

*(Note: this needs an internet connection since it hits a live API)*


In [34]:
# 9(a) - flatten each user's nested dict into a single level dict

def flatten_user(user: dict) -> dict:
    flat = {
        "id": user["id"],
        "name": user["name"],
        "username": user["username"],
        "email": user["email"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "lat": user["address"]["geo"]["lat"],
        "lng": user["address"]["geo"]["lng"],
        "company_name": user["company"]["name"],
        "company_catchphrase": user["company"]["catchPhrase"]
    }
    return flat

users_response_3 = requests.get("https://jsonplaceholder.typicode.com/users")
users_list_3 = users_response_3.json()

flattened_users = [flatten_user(u) for u in users_list_3]

for fu in flattened_users:
    print(fu)


{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'lat': '-37.3159', 'lng': '81.1496', 'company_name': 'Romaguera-Crona', 'company_catchphrase': 'Multi-layered client-server neural-net'}
{'id': 2, 'name': 'Ervin Howell', 'username': 'Antonette', 'email': 'Shanna@melissa.tv', 'city': 'Wisokyburgh', 'zipcode': '90566-7771', 'lat': '-43.9509', 'lng': '-34.4618', 'company_name': 'Deckow-Crist', 'company_catchphrase': 'Proactive didactic contingency'}
{'id': 3, 'name': 'Clementine Bauch', 'username': 'Samantha', 'email': 'Nathan@yesenia.net', 'city': 'McKenziehaven', 'zipcode': '59590-4157', 'lat': '-68.6102', 'lng': '-47.0653', 'company_name': 'Romaguera-Jacobson', 'company_catchphrase': 'Face to face bifurcated interface'}
{'id': 4, 'name': 'Patricia Lebsack', 'username': 'Karianne', 'email': 'Julianne.OConner@kory.org', 'city': 'South Elvis', 'zipcode': '53919-4257', 'lat': '29.4572', 'lng': '-164.2990', '

In [35]:
# 9(b) - DataFrame + distance from origin

df_users = pd.DataFrame(flattened_users)
print(df_users)
print()

df_users["lat"] = df_users["lat"].astype(float)
df_users["lng"] = df_users["lng"].astype(float)

# computing distance row by row using math.sqrt
distances = []
for index, row in df_users.iterrows():
    d = math.sqrt(row["lat"]**2 + row["lng"]**2)
    distances.append(d)

df_users["distance_from_origin"] = distances

farthest_user = df_users.loc[df_users["distance_from_origin"].idxmax()]
print("User with the greatest distance from origin:")
print(farthest_user)


   id                      name          username                      email  \
0   1             Leanne Graham              Bret          Sincere@april.biz   
1   2              Ervin Howell         Antonette          Shanna@melissa.tv   
2   3          Clementine Bauch          Samantha         Nathan@yesenia.net   
3   4          Patricia Lebsack          Karianne  Julianne.OConner@kory.org   
4   5          Chelsey Dietrich            Kamren   Lucio_Hettinger@annie.ca   
5   6      Mrs. Dennis Schulist  Leopoldo_Corkery    Karley_Dach@jasper.info   
6   7           Kurtis Weissnat      Elwyn.Skiles     Telly.Hoeger@billy.biz   
7   8  Nicholas Runolfsdottir V     Maxime_Nienow       Sherwood@rosamond.me   
8   9           Glenna Reichert          Delphine    Chaim_McDermott@dana.io   
9  10        Clementina DuBuque    Moriah.Stanton     Rey.Padberg@karina.biz   

             city     zipcode       lat        lng        company_name  \
0     Gwenborough  92998-3874  -37.3159    81

In [36]:
# 9(c) - hemisphere grouping

northern_users = df_users[df_users["lat"] > 0]
southern_users = df_users[df_users["lat"] < 0]

print("Northern Hemisphere users:", list(northern_users["name"]))
print("Southern Hemisphere users:", list(southern_users["name"]))

north_avg_dist = northern_users["distance_from_origin"].mean()
south_avg_dist = southern_users["distance_from_origin"].mean()

print()
print("Northern avg distance:", round(north_avg_dist, 2))
print("Southern avg distance:", round(south_avg_dist, 2))

if north_avg_dist > south_avg_dist:
    print("Northern Hemisphere has the higher average distance.")
else:
    print("Southern Hemisphere has the higher average distance.")


Northern Hemisphere users: ['Patricia Lebsack', 'Kurtis Weissnat', 'Glenna Reichert']
Southern Hemisphere users: ['Leanne Graham', 'Ervin Howell', 'Clementine Bauch', 'Chelsey Dietrich', 'Mrs. Dennis Schulist', 'Nicholas Runolfsdottir V', 'Clementina DuBuque']

Northern avg distance: 123.58
Southern avg distance: 84.32
Northern Hemisphere has the higher average distance.


In [37]:
# 9(d) - comment counts per post

posts_response_2 = requests.get("https://jsonplaceholder.typicode.com/posts")
comments_response = requests.get("https://jsonplaceholder.typicode.com/comments")

posts_list_2 = posts_response_2.json()
comments_list = comments_response.json()

post_comment_counts = []

for post in posts_list_2:
    post_id = post["id"]
    # count how many comments have this postId
    count = 0
    for c in comments_list:
        if c["postId"] == post_id:
            count += 1
    post_comment_counts.append((post_id, post["title"][:40], count))

# sort descending by comment count
post_comment_counts.sort(key=lambda x: x[2], reverse=True)

print("Top 5 most commented posts:")
for item in post_comment_counts[:5]:
    print(item)

print()
print("Bottom 5 least commented posts:")
for item in post_comment_counts[-5:]:
    print(item)


Top 5 most commented posts:
(1, 'sunt aut facere repellat provident occae', 5)
(2, 'qui est esse', 5)
(3, 'ea molestias quasi exercitationem repell', 5)
(4, 'eum et est occaecati', 5)
(5, 'nesciunt quas odio', 5)

Bottom 5 least commented posts:
(96, 'quaerat velit veniam amet cupiditate aut', 5)
(97, 'quas fugiat ut perspiciatis vero provide', 5)
(98, 'laboriosam dolor voluptates', 5)
(99, 'temporibus sit alias delectus eligendi p', 5)
(100, 'at nam consequatur ea labore ea harum', 5)
